# 1. Imports and Paths

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "fx_fintech_product_analytics.db"

DB_PATH

# 2. Connect and Check Table Columns

In [ ]:
conn = sqlite3.connect(DB_PATH)

columns = pd.read_sql_query("""
    PRAGMA table_info(modeling_user_month_snapshots);
""", conn)

columns

# 3. Table Summary

In [ ]:
summary = pd.read_sql_query("""
    SELECT
        COUNT(*) AS snapshot_rows,
        COUNT(DISTINCT user_id) AS users,
        MIN(observation_date) AS min_observation_date,
        MAX(observation_date) AS max_observation_date,
        SUM(target_repeat_30d) AS positive_targets,
        ROUND(
            100.0 * SUM(target_repeat_30d) / COUNT(*),
            2
        ) AS positive_target_rate,
        ROUND(AVG(recency_days), 2) AS avg_recency_days,
        ROUND(AVG(transactions_90d), 2) AS avg_transactions_90d,
        ROUND(AVG(failed_ratio_90d), 4) AS avg_failed_ratio_90d
    FROM modeling_user_month_snapshots;
""", conn)

summary

# 4. Preview Rows

In [ ]:
sample = pd.read_sql_query("""
    SELECT *
    FROM modeling_user_month_snapshots
    LIMIT 10;
""", conn)

sample

# 5. Missing Values

In [ ]:
df = pd.read_sql_query("""
    SELECT *
    FROM modeling_user_month_snapshots;
""", conn)

conn.close()

missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing_summary["missing_rate"] = missing_summary["missing_count"] / len(df)

missing_summary.sort_values("missing_count", ascending=False)

# 6. Target Balance

In [ ]:
target_balance = (
    df["target_repeat_30d"]
    .value_counts(normalize=False)
    .rename_axis("target_repeat_30d")
    .reset_index(name="rows")
)

target_balance["share"] = target_balance["rows"] / target_balance["rows"].sum()

target_balance

# 7. Date Coverage

In [ ]:
df["observation_date"] = pd.to_datetime(df["observation_date"])

monthly_rows = (
    df.groupby(df["observation_date"].dt.to_period("M"))
    .agg(
        rows=("user_id", "count"),
        users=("user_id", "nunique"),
        positive_rate=("target_repeat_30d", "mean"),
    )
    .reset_index()
)

monthly_rows["observation_month"] = monthly_rows["observation_date"].astype(str)
monthly_rows = monthly_rows.drop(columns="observation_date")

monthly_rows.head(), monthly_rows.tail()

# 8. Inspect Columns

In [ ]:
df.columns.tolist()

# 9. Basic Modeling Dataset

In [ ]:
target_col = "target_repeat_30d"

id_cols = [
    "user_id",
    "observation_date",
]

feature_cols = [
    col for col in df.columns
    if col not in id_cols + [target_col]
]

feature_cols

# 10. Check Feature Types

In [ ]:
df[feature_cols].dtypes.sort_values()

# 11. Time-Based Train/Test Split

In [ ]:
df = df.sort_values("observation_date").copy()

cutoff_date = df["observation_date"].quantile(0.8)

train_df = df[df["observation_date"] <= cutoff_date].copy()
test_df = df[df["observation_date"] > cutoff_date].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Cutoff date:", cutoff_date)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

# 12. Missing Value Check for Selected Features

In [ ]:
missing_features = (
    X_train.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_count"})
)

missing_features["missing_rate"] = missing_features["missing_count"] / len(X_train)
missing_features.sort_values("missing_count", ascending=False).head(20)

# 13. Rebuild Split with Explicit Cutoff

In [ ]:
target_col = "target_repeat_30d"

feature_cols = [
    "recency_days",
    "transactions_90d",
    "transactions_all",
    "total_volume_90d",
    "failed_ratio_90d",
]

cutoff_date = pd.Timestamp("2025-10-31")

train_df = df[df["observation_date"] <= cutoff_date].copy()
test_df = df[df["observation_date"] > cutoff_date].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

# 14. Baseline Model - Majority Class

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

baseline_pred = np.zeros(len(y_test))

baseline_metrics = {
    "model": "Majority class baseline",
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0),
    "roc_auc": 0.5,
}

baseline_metrics

# 15. Logistic Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

logreg_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_test)
logreg_proba = logreg_model.predict_proba(X_test)[:, 1]

logreg_metrics = {
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, logreg_pred),
    "precision": precision_score(y_test, logreg_pred),
    "recall": recall_score(y_test, logreg_pred),
    "f1": f1_score(y_test, logreg_pred),
    "roc_auc": roc_auc_score(y_test, logreg_proba),
}

logreg_metrics

# 16. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "model": "Random Forest",
    "accuracy": accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred),
    "recall": recall_score(y_test, rf_pred),
    "f1": f1_score(y_test, rf_pred),
    "roc_auc": roc_auc_score(y_test, rf_proba),
}

rf_metrics

# 18. Gradient Boosting Model

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_metrics = {
    "model": "XGBoost",
    "accuracy": accuracy_score(y_test, xgb_pred),
    "precision": precision_score(y_test, xgb_pred),
    "recall": recall_score(y_test, xgb_pred),
    "f1": f1_score(y_test, xgb_pred),
    "roc_auc": roc_auc_score(y_test, xgb_proba),
}

xgb_metrics

# Compare Models

In [ ]:
model_metrics = pd.DataFrame([
    baseline_metrics,
    logreg_metrics,
    rf_metrics,
    xgb_metrics,
])

model_metrics.sort_values("roc_auc", ascending=False)

# 20. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_pred,
    display_labels=["No repeat", "Repeat"],
    cmap="Blues",
    ax=ax,
)

ax.set_title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.show()

# 21. Classification Report

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    xgb_pred,
    target_names=["No repeat", "Repeat"],
    output_dict=True,
)

classification_report_df = pd.DataFrame(report).T

classification_report_df

# 24. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

XGBoost is selected as the final model because it provides the strongest F1 score while maintaining high recall, making it useful for identifying users likely to repeat exchange activity in the next 30 days.